# Human in the Loop

Exploring the ability for HITL verification fo actions by the agent

In [1]:
from langchain.agents import create_agent, AgentState
from langchain.messages import HumanMessage
from langchain.tools import tool, ToolRuntime
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from dotenv import load_dotenv

from pprint import pprint

In [2]:
load_dotenv()

True

In [3]:
@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address"""
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send the drafted email to the given address"""
    return f"Email sent"

Setting up the agent state and creating the agent with the HITL middleware. This middleware specifies which tool calls require human input before executing.

In [6]:
# Setting up the Agent's State
class EmailState(AgentState):
    email: str

agent = create_agent(
    model="claude-haiku-4-5",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware[AgentState, None](
            interrupt_on={
                "read_email": False,
                "send_email": True
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
    system_prompt="""When asked to send an email, always call send_email 
    directly with your best draft. Do not ask the user for confirmation in 
    the chat — a separate approval step already exists."""
)

In [7]:
config = {"configurable": {"thread_id": "1"}}

### Approving

Approving the HITL interrupted action reqiures invoking the agent with a Command object specifying the decision. In this case it receives a tuple with a named dict called "resume" and the included decisions for the agent. This requires memory and passing the same config thread.

In [8]:
approval = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email, draft a response and send it")],
        "email": "Hi Sam, can I add a topic to the agenda for our meeting tomorrow?",
    },
    config=config
)

In [9]:
pprint(approval)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi,\n'
                                                                          '\n'
                                                                          'Of '
                                                                          'course! '
                                                                          "I'd "
                                                                          'be '
                                                                          'happy '
                                                                          'to '
                                                                          'add '
                                                                          'a '
                                                                          'topic '
                                                                          'to '
                           

In [10]:
response = agent.invoke(
    Command[tuple[()]](
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config
)

pprint(response)

{'email': 'Hi Sam, can I add a topic to the agenda for our meeting tomorrow?',
 'messages': [HumanMessage(content='Please read my email, draft a response and send it', additional_kwargs={}, response_metadata={}, id='897a4e01-b9fa-4820-8605-e445ee120c39'),
              AIMessage(content=[{'text': "I'll read your email first.", 'type': 'text'}, {'id': 'toolu_01Sk5Am6HT6Vc65WX1WYcJkK', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_email', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011Cedi4TS885n1gbMeR6C2h', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 654, 'output_tokens': 44, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'},

### Rejecting

Rejecting the tool call is very similar -- pass a command with a reject decision. This is clunky as the agent will request another approval of the tool call before executing.

In [11]:
config2 = {"configurable": {"thread_id": "2"}}

reject = agent.invoke(
    {
        "messages":[HumanMessage(content="Please read my email, draft a response and send it")],
        "email": "Hi Sam, can I add a topic to the agenda for our meeting tomorrow?",
    },
    config=config2
)

In [12]:
pprint(reject)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi,\n'
                                                                          '\n'
                                                                          'Of '
                                                                          'course! '
                                                                          "I'd "
                                                                          'be '
                                                                          'happy '
                                                                          'to '
                                                                          'add '
                                                                          'a '
                                                                          'topic '
                                                                          'to '
                           

In [13]:
reject = agent.invoke(
    Command[tuple[()]](
        resume={
            "decisions": [
                {
                    "type": "reject",
                    "message": "Don't send this reply. Draft a response that asks the sender if they'd like to speak about the agent project they've been working on."
                }
            ]
        }
    ),
    config=config2
)

In [15]:
pprint(reject)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi,\n'
                                                                          '\n'
                                                                          'Of '
                                                                          'course! '
                                                                          "I'd "
                                                                          'be '
                                                                          'happy '
                                                                          'to '
                                                                          'add '
                                                                          'a '
                                                                          'topic '
                                                                          'to '
                           

### Editing

Editing removes the clunkiness introduced by rejecting -- it changes the behavior of the agent and lets it proceed with execution without a second approval.

In [16]:
config3 = {"configurable": {"thread_id": "3"}}

edit = agent.invoke(
    {
        "messages":[HumanMessage(content="Please read my email and send a response")],
        "email": "Hi Sam, can I add a topic to the agenda for our meeting tomorrow?",
    },
    config=config3
)

In [17]:
pprint(edit)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi,\n'
                                                                          '\n'
                                                                          'Of '
                                                                          'course! '
                                                                          "I'd "
                                                                          'be '
                                                                          'happy '
                                                                          'to '
                                                                          'add '
                                                                          'a '
                                                                          'topic '
                                                                          'to '
                           

In [18]:
edit = agent.invoke(
    Command[tuple[()]](
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "send_email",
                        "args": {"body": "Yes, let's discuss the new project on agent workflows."}
                    }
                }
            ]
        }
    ),
    config=config3
)

In [19]:
edit

{'messages': [HumanMessage(content='Please read my email and send a response', additional_kwargs={}, response_metadata={}, id='173cd87c-1cf7-4d33-8b6f-3db8541b0413'),
  AIMessage(content=[{'text': "I'll read your email first.", 'type': 'text'}, {'id': 'toolu_016WzecorRevQ6Qd3tRiZC4F', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_email', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011Cedi7S28MaXdyWoc6a23V', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 651, 'output_tokens': 44, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--01a05fa2-9d83